In [117]:
import requests
import pandas as pd
import json
import time

In [118]:
metro_areas= requests.get('https://api.datausa.io/attrs/geo/?sumlevel=msa').json()
metro_areas = metro_areas['data']


In [119]:
metro_names={}
for data in metro_areas:
    metro_names[data[8]]=data[9]

In [120]:
#variabe to treat get requests as web browser requests to avoid denials
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/56.0.2924.76 Safari/537.36'}
#request the population data for all the metro areas
pop=requests.get('http://db.datausa.io/api/?show=geo&sumlevel=msa&required=pop&year=all',headers=headers).json()['data']
#adds Metro Area names
for i in pop:
    if i[1] in metro_names.keys():
        i.append(metro_names[i[1]])

In [121]:
#convert list to Data frame
dfpop=pd.DataFrame(pop,columns=['Year','Code','Population','Place'])
#sort data by population and take 10 largest populations
dfnames=dfpop.sort_values(by=['Year','Population']).tail(10)
dfnames.reset_index()

,index,Year,Code,Population,Place
0,2895,2016,31000US14460,4728844,"Boston-Cambridge-Newton, MA-NH Metro Area"
1,2836,2016,31000US12060,5612777,"Atlanta-Sandy Springs-Roswell, GA Metro Area"
2,3345,2016,31000US33100,5926955,"Miami-Fort Lauderdale-West Palm Beach, FL Metr..."
3,3691,2016,31000US47900,6011752,"Washington-Arlington-Alexandria, DC-VA-MD-WV"
4,3461,2016,31000US37980,6047721,"Philadelphia-Camden-Wilmington, PA-NJ-DE-MD"
5,3181,2016,31000US26420,6482592,"Houston-The Woodlands-Sugar Land, TX Metro Area"
6,3004,2016,31000US19100,6957123,"Dallas-Fort Worth-Arlington, TX"
7,2954,2016,31000US16980,9528396,"Chicago-Naperville-Elgin, IL-IN-WI Metro Area"
8,3293,2016,31000US31080,13189366,"Los Angeles-Long Beach-Anaheim, CA Metro Area"
9,3403,2016,31000US35620,20031443,"New York-Newark-Jersey City, NY-NJ-PA Metro Area"


In [122]:
#median property value
mpv=requests.get('https://db.datausa.io/api/?show=geo&sumlevel=msa&required=median_property_value&year=all',headers=headers).json()['data']
#commute time
commute_time=requests.get('https://datausa.io/api/data?measure=Average Commute Time&drilldowns=MSA',headers=headers).json()['data']
mpvtemp=[]
for lis in mpv:
     if lis[1] in dfnames['Code'].values:
            lis.append(metro_names[lis[1]])
            mpvtemp.append(lis)
df_mpv=pd.DataFrame(mpvtemp,columns=['Year','Code','Median Property Value','Place'])
df_mpv=df_mpv.sort_values(['Place','Year'])
#df_mpv=df_mpv.set_index(['Place','Year'])
df_mpv.head()

,Year,Code,Median Property Value,Place
0,2013,31000US12060,170400.0,"Atlanta-Sandy Springs-Roswell, GA Metro Area"
10,2014,31000US12060,167400.0,"Atlanta-Sandy Springs-Roswell, GA Metro Area"
20,2015,31000US12060,168100.0,"Atlanta-Sandy Springs-Roswell, GA Metro Area"
30,2016,31000US12060,173300.0,"Atlanta-Sandy Springs-Roswell, GA Metro Area"
1,2013,31000US14460,362400.0,"Boston-Cambridge-Newton, MA-NH Metro Area"


In [123]:
commute_tmp=[]
columns=['ID MSA','Slug MSA','Year','Average Commute Time']
for lis in commute_time:
     if lis['ID MSA'] in dfnames['Code'].values:
            temp=[]
            for i in columns:
                temp.append(lis[i])
            commute_tmp.append(temp)
df_commute=pd.DataFrame(commute_tmp,columns=['Code','Place','Year','Average Commute Time'])
df_commute=df_commute.sort_values(['Place','Year']).reset_index(drop=True)
#df_commute.set_index(['Place','Year'])

In [125]:
df_pop=dfpop[dfpop.Code.isin(dfnames.Code.values)].reset_index(drop=True)

In [224]:
df=df_pop.join(df_mpv,lsuffix='Code', rsuffix='Code')
df = df.loc[:,~df.columns.duplicated()]
df.columns=['Year','Code','Population','Place','Median Property Value']
df=df.join(df_commute,lsuffix='Code',rsuffix='Code')
df = df.loc[:,~df.columns.duplicated()]
df.columns=['Year','Code','Population','Geography','Median Property Value','Average Commute Time']
df=df.drop('Code',axis=1).sort_values(['Geography','Year']).set_index(['Geography','Year'])
df.head()

Population  \
Geography                                    Year               
Atlanta-Sandy Springs-Roswell, GA Metro Area 2013     5379176   
                                             2014     5455053   
                                             2015     5535837   
                                             2016     5612777   
Boston-Cambridge-Newton, MA-NH Metro Area    2013     4604278   

                                                   Median Property Value  \
Geography                                    Year                          
Atlanta-Sandy Springs-Roswell, GA Metro Area 2013               170400.0   
                                             2014               167400.0   
                                             2015               168100.0   
                                             2016               173300.0   
Boston-Cambridge-Newton, MA-NH Metro Area    2013               362400.0   

                                                   Average Commute Time  
Geography                                    Year                        
Atlanta-Sandy Springs-Roswell, GA Metro Area 2013             28.203400  
                                             2014             29.440874  
                                             2015             28.067547  
                                             2016             26.402195  
Boston-Cambridge-Newton, MA-NH Metro Area    2013             29.101936

In [128]:
commute_mode={}
poverty={}
ownership={}
income={}
age={}
property_tax={}
citizen={}
local={}
for i in range(10):
    commute_mode[dfnames.iloc[i]['Place']]=requests.get('https://datausa.io/api/data?measure=Commute%20Means,Commute%20Means%20Moe&geo='+dfnames.iloc[i]['Code']+'&drilldowns=Group',headers=headers).json()['data']
    time.sleep(5)
    poverty[dfnames.iloc[i]['Place']]=requests.get('https://datausa.io/api/data?Geography='+dfnames.iloc[i]['Code'] +'&measure=Poverty%20Population,Poverty%20Population%20Moe&Poverty%20Status=0',headers=headers).json()['data']
    time.sleep(5)
    ownership[dfnames.iloc[i]['Place']]=requests.get('https://datausa.io/api/data?measure=Household%20Ownership,Household%20Ownership%20Moe&Geography='+dfnames.iloc[i]['Code']+'&drilldowns=Occupied%20By',headers=headers).json()['data']
    time.sleep(5)
    income[dfnames.iloc[i]['Place']]=requests.get('https://datausa.io/api/data?Geography='+dfnames.iloc[i]['Code']+'&measure=Total%20Population,Total%20Population%20MOE%20Appx,Record%20Count&drilldowns=Wage%20Bin&Workforce%20Status=true&Record%20Count%3E=5',headers=headers).json()['data']
    time.sleep(5)
    age[dfnames.iloc[i]['Place']]=requests.get('https://datausa.io/api/data?Geography='+dfnames.iloc[i]['Code']+'&measures=Birthplace,Birthplace%20Moe&drilldowns=Place%20of%20Birth,Age',headers=headers).json()['data']
    time.sleep(5)
    property_tax[dfnames.iloc[i]['Place']]=requests.get('https://datausa.io/api/data?measure=Real%20Estate%20Taxes%20by%20Mortgage,Real%20Estate%20Taxes%20by%20Mortgage%20Moe&drilldowns=Real%20Estate%20Taxes%20Paid&geo='+dfnames.iloc[i]['Code'],headers=headers).json()['data']
    time.sleep(5)
    citizen[dfnames.iloc[i]['Place']]=requests.get('https://datausa.io/api/data?measure=Citizenship%20Status&drilldowns=Citizenship&Geography='+dfnames.iloc[i]['Code'],headers=headers).json()['data']
    time.sleep(5)
    local[dfnames.iloc[i]['Place']]=requests.get('https://datausa.io/api/data?measure=Foreign-Born%20Citizens,Population&Geography='+dfnames.iloc[i]['Code'],headers=headers).json()['data']

In [256]:
#resests vairable if cell is rerun so it doesn't try to doube data
dfcommute_mode=None
#convert commute mode data from dictionaries to data frame
for name in commute_mode:
    for i in range(len(commute_mode[name])):
        dfcommute_mode=pd.concat([dfcommute_mode, pd.DataFrame.from_dict(commute_mode[name][i],orient='index').T ])
#remove extra columns
dfcommute_mode=dfcommute_mode.drop(['ID Geography','Slug Geography','ID Year'],axis=1)
#reindex by location and year
dfcommute_mode=dfcommute_mode.sort_values(['Geography','Year','ID Group']).set_index(['Geography','Year'])
# fix data types
dfcommute_mode['ID Group']=dfcommute_mode['ID Group'].astype(int)
dfcommute_mode['Commute Means']=dfcommute_mode['Commute Means'].astype(int)
dfcommute_mode['Commute Means Moe']=dfcommute_mode['Commute Means Moe'].astype(float)
dfcommute_mode.head()

ID Group           Group  \
Geography                         Year                             
Atlanta-Sandy Springs-Roswell, GA 2013         0     Drove Alone   
                                  2013         1       Carpooled   
                                  2013         2  Public Transit   
                                  2013         3            Taxi   
                                  2013         4      Motorcycle   

                                        Commute Means  Commute Means Moe  
Geography                         Year                                    
Atlanta-Sandy Springs-Roswell, GA 2013        1958050       19610.000000  
                                  2013         261745       11502.365800  
                                  2013          77644        5531.767439  
                                  2013           4572        1535.000000  
                                  2013           2886         634.000000

In [255]:
dfpoverty=None
for name in poverty:
    for i in range(len(poverty[name])):
        dfpoverty=pd.concat([dfpoverty, pd.DataFrame.from_dict(poverty[name][i],orient='index').T ])
dfpoverty=dfpoverty.drop(['ID Geography','Slug Geography','ID Year','ID Poverty Status','Poverty Status'],axis=1)
#reindex by location and year
dfpoverty=dfpoverty.sort_values(['Geography','Year']).set_index(['Geography','Year'])
#fix data types
dfpoverty['Poverty Population']=dfpoverty['Poverty Population'].astype(int)
dfpoverty['Poverty Population Moe']=dfpoverty['Poverty Population Moe'].astype(float)
dfpoverty.head()

Poverty Population  \
Geography                         Year                       
Atlanta-Sandy Springs-Roswell, GA 2013              809682   
                                  2014              840560   
                                  2015              847748   
                                  2016              820456   
                                  2017              780843   

                                        Poverty Population Moe  
Geography                         Year                          
Atlanta-Sandy Springs-Roswell, GA 2013             7442.206528  
                                  2014             7441.942152  
                                  2015             7655.603307  
                                  2016             7661.686368  
                                  2017             7497.033147

In [238]:
dfownership=None
for name in ownership:
    for i in range(len(ownership[name])):
        dfownership=pd.concat([dfownership, pd.DataFrame.from_dict(ownership[name][i],orient='index').T ])
dfownership=dfownership.drop(['ID Geography','Slug Geography','ID Year'],axis=1)
#reindex by location and year
dfownership=dfownership.sort_values(['Geography','Year','ID Occupied By']).set_index(['Geography','Year'])
dfownership['Household Ownership']=dfownership['Household Ownership'].astype(int)
dfownership['Household Ownership Moe']=dfownership['Household Ownership Moe'].astype(int)
dfownership.head()

ID Occupied By      Occupied By  \
Geography                         Year                                   
Atlanta-Sandy Springs-Roswell, GA 2013              0   Owner Occupied   
                                  2013              1  Renter Occupied   
                                  2014              0   Owner Occupied   
                                  2014              1  Renter Occupied   
                                  2015              0   Owner Occupied   

                                        Household Ownership  \
Geography                         Year                        
Atlanta-Sandy Springs-Roswell, GA 2013              1237449   
                                  2013               716899   
                                  2014              1236658   
                                  2014               744789   
                                  2015              1248726   

                                        Household Ownership Moe  
Geography                         Year                           
Atlanta-Sandy Springs-Roswell, GA 2013                    12058  
                                  2013                    11472  
                                  2014                    12105  
                                  2014                    11717  
                                  2015                    13043

In [286]:
dfincome=None
for name in income:
    for i in range(84):
        temp=pd.DataFrame.from_dict(income[name][i],orient='index').T
        temp['Geography']=name
        dfincome=pd.concat([dfincome, temp])
dfincome=dfincome.drop(['ID Geography','ID Year', 'ID Workforce Status','Workforce Status'],axis=1)
#reindex by location and year
dfincome=dfincome.sort_values(['Geography','Year','ID Wage Bin']).set_index(['Geography','Year'])
dfincome['ID Wage Bin']=dfincome['ID Wage Bin'].astype(int)
dfincome['Total Population']=dfincome['Total Population'].astype(int)
dfincome['Total Population MOE Appx']=dfincome['Total Population MOE Appx'].astype(float)
dfincome.head()

ID Wage Bin Wage Bin  \
Geography                                    Year                         
Atlanta-Sandy Springs-Roswell, GA Metro Area 2014            1   < $10K   
                                             2014            2  $10-20k   
                                             2014            3  $20-30k   
                                             2014            4  $30-40k   
                                             2014            5  $40-50k   

                                                   Total Population  \
Geography                                    Year                     
Atlanta-Sandy Springs-Roswell, GA Metro Area 2014            591714   
                                             2014            680354   
                                             2014            675005   
                                             2014            592683   
                                             2014            442866   

                                                   Total Population MOE Appx  \
Geography                                    Year                              
Atlanta-Sandy Springs-Roswell, GA Metro Area 2014               17529.625818   
                                             2014               18569.198814   
                                             2014               18509.818465   
                                             2014               17541.664860   
                                             2014               15468.837438   

                                                  Record Count  
Geography                                    Year               
Atlanta-Sandy Springs-Roswell, GA Metro Area 2014         5419  
                                             2014         6059  
                                             2014         6058  
                                             2014         5335  
                                             2014         4202

In [280]:
dfage=None
for name in age:
    for i in range(220):
        dfage=pd.concat([dfage, pd.DataFrame.from_dict(age[name][i],orient='index').T ])
dfage=dfage.drop(['ID Geography','Slug Geography','ID Year'],axis=1)
#reindex by location and year
dfage=dfage.sort_values(['Geography','Year','ID Place of Birth','ID Age']).set_index(['Geography','Year'])
dfage['ID Place of Birth']=dfage['ID Place of Birth'].astype(int)
dfage['ID Age']=dfage['ID Age'].astype(int)
dfage['Birthplace']=dfage['Birthplace'].astype(int)
dfage['Birthplace Moe']=dfage['Birthplace Moe'].astype(int)
dfage.head(44)

ID Place of Birth  \
Geography                         Year                      
Atlanta-Sandy Springs-Roswell, GA 2013                  0   
                                  2013                  0   
                                  2013                  0   
                                  2013                  0   
                                  2013                  0   
                                  2013                  0   
                                  2013                  0   
                                  2013                  0   
                                  2013                  0   
                                  2013                  0   
                                  2013                  0   
                                  2013                  1   
                                  2013                  1   
                                  2013                  1   
                                  2013                  1   
                                  2013                  1   
                                  2013                  1   
                                  2013                  1   
                                  2013                  1   
                                  2013                  1   
                                  2013                  1   
                                  2013                  1   
                                  2013                  2   
                                  2013                  2   
                                  2013                  2   
                                  2013                  2   
                                  2013                  2   
                                  2013                  2   
                                  2013                  2   
                                  2013                  2   
                                  2013                  2   
                                  2013                  2   
                                  2013                  2   
                                  2013                  3   
                                  2013                  3   
                                  2013                  3   
                                  2013                  3   
                                  2013                  3   
                                  2013                  3   
                                  2013                  3   
                                  2013                  3   
                                  2013                  3   
                                  2013                  3   
                                  2013                  3   

                                                                  Place of Birth  \
Geography                         Year                                             
Atlanta-Sandy Springs-Roswell, GA 2013                Born In State of Residence   
                                  2013                Born In State of Residence   
                                  2013                Born In State of Residence   
                                  2013                Born In State of Residence   
                                  2013                Born In State of Residence   
                                  2013                Born In State of Residence   
                                  2013                Born In State of Residence   
                                  2013                Born In State of Residence   
                                  2013                Born In State of Residence   
                                  2013                Born In State of Residence   
                                  2013                Born In State of Residence   
                                  2013  Born In Other State In The United States   
                                  2013  Born In Other Sta

In [270]:
dfproperty_tax=None
for name in property_tax:
    for i in range(30):
        dfproperty_tax=pd.concat([dfproperty_tax, pd.DataFrame.from_dict(property_tax[name][i],orient='index').T ])
dfproperty_tax=dfproperty_tax.drop(['ID Geography','Slug Geography','ID Year'],axis=1)
#reindex by location and year
dfproperty_tax=dfproperty_tax.sort_values(['Geography','Year','ID Real Estate Taxes Paid']).set_index(['Geography','Year'])
dfproperty_tax['ID Real Estate Taxes Paid']=dfproperty_tax['ID Real Estate Taxes Paid'].astype(int)
dfproperty_tax['Real Estate Taxes by Mortgage']=dfproperty_tax['Real Estate Taxes by Mortgage'].astype(int)
dfproperty_tax['Real Estate Taxes by Mortgage Moe']=dfproperty_tax['Real Estate Taxes by Mortgage Moe'].astype(float)
dfproperty_tax.head(6)


ID Real Estate Taxes Paid  \
Geography                         Year                              
Atlanta-Sandy Springs-Roswell, GA 2013                          0   
                                  2013                          1   
                                  2013                          2   
                                  2013                          3   
                                  2013                          4   
                                  2013                          5   

                                           Real Estate Taxes Paid  \
Geography                         Year                              
Atlanta-Sandy Springs-Roswell, GA 2013             Less Than $800   
                                  2013             $800 to $1,499   
                                  2013           $1,500 to $1,999   
                                  2013           $2,000 to $2,999   
                                  2013             $3,000 or More   
                                  2013  No Real Estate Taxes Paid   

                                        Real Estate Taxes by Mortgage  \
Geography                         Year                                  
Atlanta-Sandy Springs-Roswell, GA 2013                         272928   
                                  2013                         307872   
                                  2013                         157570   
                                  2013                         206166   
                                  2013                         271118   
                                  2013                          21795   

                                        Real Estate Taxes by Mortgage Moe  
Geography                         Year                                     
Atlanta-Sandy Springs-Roswell, GA 2013                        8392.255120  
                                  2013                        9155.838192  
                                  2013                        6713.669637  
                                  2013                        7590.195913  
                                  2013                        7747.776584  
                                  2013                        2422.068744

In [282]:
dfcitizen=None
for name in citizen:
    for i in range(10):
        dfcitizen=pd.concat([dfcitizen, pd.DataFrame.from_dict(citizen[name][i],orient='index').T ])
dfcitizen=dfcitizen.drop(['ID Geography','Slug Geography','ID Year'],axis=1)
#reindex by location and year
dfcitizen=dfcitizen.sort_values(['Geography','Year','ID Citizenship']).set_index(['Geography','Year'])
dfcitizen['ID Citizenship']=dfcitizen['ID Citizenship'].astype(int)
dfcitizen['Citizenship Status']=dfcitizen['Citizenship Status'].astype(int)
dfcitizen.head()

ID Citizenship  Citizenship  \
Geography                         Year                                
Atlanta-Sandy Springs-Roswell, GA 2013               0      Citizen   
                                  2013               1  Non-Citizen   
                                  2014               0      Citizen   
                                  2014               1  Non-Citizen   
                                  2015               0      Citizen   

                                        Citizenship Status  
Geography                         Year                      
Atlanta-Sandy Springs-Roswell, GA 2013             5089192  
                                  2013              435501  
                                  2014             5186771  
                                  2014              425058  
                                  2015             5265746

In [288]:
dflocal=None
for name in local:
    for i in range(5):
        dflocal=pd.concat([dflocal, pd.DataFrame.from_dict(local[name][i],orient='index').T ])
dflocal=dflocal.drop(['ID Geography','Slug Geography','ID Year'],axis=1)
#reindex by location and year
dflocal=dflocal.sort_values(['Geography','Year']).set_index(['Geography','Year'])

C:\Users\aleja\Anaconda3\lib\site-packages\ipykernel_launcher.py:4: FutureWarning: Sorting because non-concatenation axis is not aligned. A future version
of pandas will change to not sort by default.

To accept the future behavior, pass 'sort=False'.

To retain the current behavior and silence the warning, pass 'sort=True'.

  after removing the cwd from sys.path.


In [289]:
print(dflocal.info())
dflocal.head()


<class 'pandas.core.frame.DataFrame'>
MultiIndex: 50 entries, (Atlanta-Sandy Springs-Roswell, GA, 2013) to (Washington-Arlington-Alexandria, DC-VA-MD-WV, 2017)
Data columns (total 2 columns):
Foreign-Born Citizens    30 non-null object
Population               50 non-null object
dtypes: object(2)
memory usage: 1.1+ KB
None


Foreign-Born Citizens Population
Geography                         Year                                 
Atlanta-Sandy Springs-Roswell, GA 2013                   NaN    5524693
                                  2014                   NaN    5611829
                                  2015                743009    5709731
                                  2016                753532    5790210
                                  2017                777555    5882450